<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>

# 4. RT-Embed: Video Search

RT-Embed (video embeddings) uses Cosmos-Embed1 to turn each 5-second chunk of video into a vector. A sentence
becomes a vector in the same space, so you can find moments in hours of video by describing them. Here you search
four traffic videos at once and watch only the seconds that match.

In this notebook you will:

- turn a sentence into a vector and find the nearest video chunks across four videos
- compare a real query with an unrelated one, read the scores as a ranking, and watch the windows
- write queries the way a caption describes a moment, and measure them against labelled events
- search one video at a time
- optionally, use a short video clip as the query instead of a sentence

**Time:** 30 minutes

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **Part 1** &middot; about 30 min &nbsp;|&nbsp; [Home](../00_Home.ipynb)
>
> **Before you start:** the lab pill is green, which means the four search videos are indexed.
>
> Previous: [3. RT-VLM: Captioning & Prompting](03-rt-vlm-captioning-and-prompting.ipynb) &middot; Next: [Part 1 Assessment](assessment.ipynb)


<img src="../images/course/map-search.png" width="960" alt="The pipeline map with VIOS, RT-Embed, Kafka and Elasticsearch highlighted, and your notebook, vss-agent, RT-CV and RT-VLM greyed out. Video stored in VIOS goes to RT-Embed, and RT-Embed's output travels through Kafka into Elasticsearch.">

## 4.1 Check the prepared collection

The collection is four traffic videos of about three minutes each: `normal_eval.mp4`, `rain_eval.mp4`,
`fog_eval.mp4` and `night_eval.mp4`. Their names describe the conditions, not what happens in them. Do not assume
that "normal" means nothing happens. You will see these videos again. The coding agent searches them at the end of
Part 1. In Part 2 you deploy VSS for traffic anomalies and evaluate it on the same footage. There the files are named
`eval_video_01` (normal), `eval_video_02` (night), `eval_video_03` (fog) and `eval_video_04` (rain).

Uploading and indexing them takes too long for class time, so the `course-prep` service did it when your lab
started. The service wrote two files in `/dli/task/data/`:

- `search_sources.json` lists each video with its VIOS sensor id and the start of its VIOS timeline.
- `course_prep_status.json` records whether the preparation finished. You checked it in the onboarding notebook.

The next cell reads both files and checks each video against the VIOS sensor list. It uploads nothing. When it
passes, it lists the four videos with their sensor ids. If it reports `RUNNING`, wait a minute and run it again.
If it reports `FAILED`, call the instructor.

Two dates matter in this notebook. In VIOS, each search video's timeline starts at **2025-01-02**. In the
embedding index, every uploaded video starts at **2025-01-01**, whatever date the upload gave. The VSS agent
service sends 2025-01-01 to RT-Embed for every video it ingests. Section 4.6 shows
how the search moves a time from one to the other.

In [ ]:
import json, time, requests
from pathlib import Path
from datetime import datetime

import course_setup, labtools

BASE = "http://nginx"
TASK = Path("/dli/task")
MANIFEST = json.loads((TASK / "search-videos.json").read_text())   # the four videos, their lengths and VIOS start times
REGISTRY = TASK / "data/search_sources.json"                         # written by course-prep at launch
PREP_STATUS = TASK / "data/course_prep_status.json"                  # course-prep's state, video by video
INDEX = "mdx-embed-filtered-2025-01-01"                              # RT-Embed's vectors for uploaded videos
EMBEDDING_ANCHOR = "2025-01-01T00:00:00.000Z"                        # the date that index gives every uploaded video


def api(method, path, **kwargs):
    """One request through the course proxy, decoded as JSON. Any error status raises."""
    kwargs.setdefault("timeout", 60)
    response = requests.request(method, BASE + path, **kwargs)
    response.raise_for_status()
    return response.json()


def prep_status_text(status):
    """course-prep's state as printable lines, one per video."""
    out = [f"course-prep state: {status.get('state')}   updated: {status.get('updated')}"]
    for name, asset in status.get("assets", {}).items():
        out.append(f"  {str(asset.get('state')):8} {name:28} sensor {asset.get('sensor_id') or '-'}  {asset.get('error') or ''}")
    return "\n".join(out)


if not PREP_STATUS.is_file():
    raise FileNotFoundError(
        f"{PREP_STATUS} is missing: course-prep has not recorded anything yet. It starts once the VSS agent service "
        "is healthy, so run this cell again in a minute. Still missing ten minutes after the readiness check said "
        "READY TO CONTINUE? Call the instructor, and do not upload the videos yourself.")
prep = json.loads(PREP_STATUS.read_text())
print(prep_status_text(prep))
if prep.get("state") != "OK":
    raise RuntimeError(f"course-prep reports {prep.get('state')!r}, not OK (see the lines above). "
                       "RUNNING: run this cell again in a minute. FAILED: call the instructor.")
if not REGISTRY.is_file():
    raise FileNotFoundError(f"{REGISTRY} is missing although course-prep reports OK. Call the instructor, and do "
                            "not upload the videos yourself.")

registered = json.loads(REGISTRY.read_text())
live_ids = {s["sensorId"] for s in api("GET", "/vios/vst/api/v1/sensor/list", timeout=30)}
problems = []
for item in MANIFEST:
    entry = registered.get(item["filename"])
    if entry is None:
        problems.append(f"{item['filename']}: not in the registry")
    elif not entry.get("completed"):
        problems.append(f"{item['filename']}: the upload was never completed")
    elif entry.get("sensor_id") not in live_ids:
        problems.append(f"{item['filename']}: sensor {entry.get('sensor_id')} is not in VIOS's sensor list")
if problems:
    raise RuntimeError("the registry does not match VIOS, so searches would return nothing:\n  "
                       + "\n  ".join(problems) + "\nCall the instructor and show these lines.")

sources = {item["filename"]: registered[item["filename"]] for item in MANIFEST}
print()
for name, entry in sources.items():
    print(f"{name:18} sensor {entry['sensor_id']}  VIOS timeline starts {entry['creation_time']}")

## 4.2 Confirm each video's embeddings

An index with some records in it is not enough. It could hold only one of the videos, or only the first minute of
each. The next cell asks Elasticsearch, for each video, how many embedding records exist and where the last one
ends. It waits until every video is covered to within 10 seconds of its end.

On a prepared lab it passes on the first try. If it has to wait, it prints its progress every 10 seconds and stops
with an error after 20 minutes. This checks that indexing finished, not how good the search is.

If Elasticsearch has no records at all for the four videos, the cell stops at once with an error. The records
existed when `course-prep` reported `OK`, so they are gone. Elasticsearch deleted them under its index lifecycle
policy, the rule that sets how long it keeps an index. Call the instructor, and do not upload the videos yourself.

In [ ]:
def embedding_progress(entry):
    """(records indexed, seconds of the video covered) for one video; (0, 0.0) when the index is missing."""
    response = requests.post(BASE + "/es/" + INDEX + "/_search", timeout=30, json={
        "size": 0,
        "query": {"term": {"sensor.id.keyword": entry["sensor_id"]}},
        "aggs": {"last_end": {"max": {"field": "end"}}},
    })
    if response.status_code == 404:          # no index: not created yet, or deleted
        return 0, 0.0
    response.raise_for_status()
    data = response.json()
    total = data["hits"]["total"]["value"]
    last = data["aggregations"]["last_end"]["value"]      # epoch milliseconds, or None without records
    anchor = datetime.fromisoformat(EMBEDDING_ANCHOR.replace("Z", "+00:00")).timestamp()
    return total, (max(0.0, last / 1000 - anchor) if last is not None else 0.0)


LIMIT_S = 1200
started = time.monotonic()
while True:
    progress = {name: embedding_progress(entry) for name, entry in sources.items()}
    # course-prep says OK only once every video is searchable, so no records at all now means deleted, not late.
    if prep.get("state") == "OK" and all(count == 0 for count, _ in progress.values()):
        raise RuntimeError(
            f"course-prep reported OK, but Elasticsearch has no embedding records in {INDEX} for any of the four "
            "videos. The search records are gone: Elasticsearch deleted them under its index lifecycle policy. "
            "Call the instructor, and do not upload the videos yourself. When the instructor has prepared them "
            "again, run section 4.1 and then this cell again.")
    pending = [name for name, (count, seconds) in progress.items()
               if not count or seconds < sources[name]["duration_seconds"] - 10]
    elapsed = time.monotonic() - started
    if not pending:
        break
    if elapsed >= LIMIT_S:
        raise TimeoutError(f"embeddings still incomplete for {pending} after {elapsed:.0f} s. RT-Embed, Behavior "
                           "Analytics or Logstash is behind: call the instructor.")
    print(f"waiting for embeddings, {elapsed:.0f} s of {LIMIT_S} s: "
          + ", ".join(f"{name} {count} records to {seconds:.0f} s" for name, (count, seconds) in progress.items()),
          flush=True)
    time.sleep(10)

for name, (count, seconds) in progress.items():
    print(f"{name:18} {count:5d} embedding records, covering {seconds:.1f} s "
          f"of {sources[name]['duration_seconds']:.1f} s")

## 4.3 Encode a query

Cosmos-Embed1 puts video chunks and text in one embedding space. Its text encoder turns a sentence into 768
numbers. Search compares that vector with the stored video vectors. The numbers mean nothing on their own.

Think of each embedding as a point whose position stands for what the video or the sentence shows. The model was
trained to put matching text and video close together. So a query such as "a car stopped in the intersection" can
find a matching chunk even if no caption contains those words. The caption search in section 3.8 could only find
words that a caption happened to use.

RT-CV makes embeddings too, one for each detected object (section 2.12). Those find objects by how they look, such
as a white car. RT-Embed's vectors describe whole chunks of video, so they find moments and events, such as a
collision.

Run the next cell. It should print `Vector dimensions: 768`.

In [ ]:
r = requests.post(BASE + "/rtvi-embed/v1/generate_text_embeddings", timeout=60, json={
    "text_input": ["two vehicles collide at an intersection"],
    "model": "cosmos-embed1-448p-anomaly-detection",     # Cosmos-Embed1, the model RT-Embed serves
})
r.raise_for_status()
vector = r.json()["data"][0]["embeddings"]
print("Vector dimensions:", len(vector))
print("First 8 values:", [round(x, 4) for x in vector[:8]])

## 4.4 Search across files

A k-nearest-neighbor (kNN) search returns the `k` stored vectors closest to the query vector. Here the query is
a sentence, and its neighbors are 5-second video chunks. Each hit names a video and a window: the start and end
of its chunk. A higher score means a closer match for this query. It is not the chance that the event happened.
Treat every hit as a candidate to watch.

How `search(query, k=5)` in the next cell works:

- It asks Elasticsearch for `k * 4` neighbors, so that it still has `k` after it drops duplicates.
- `num_candidates` sets how many vectors the search looks at on each shard before it picks the nearest. A bigger
  pool can find better matches, but it takes more work.
- It filters on `sensor.id.keyword`. In the embedding index, that field holds the VIOS sensor id, not the file name.
- It removes duplicates by **video + start + end**, never by time alone, because two videos can match at the same
  second.
- Each hit keeps the file name, the sensor id and its window, both as indexed and on the video's own VIOS timeline.
- `query` can be a sentence or a vector you already have. Section 4.10 passes a vector.
- `course_setup.text_embedding` makes the same embedding request. `course_setup.knn_search` sends the same kNN
  query, but asks for only `k` neighbors, removes no duplicates and returns the raw Elasticsearch hits. The Part 1
  assessment uses both.

Start with a collision query. Each printed line is one hit: rank, score, file, and the window in seconds from the
start of that video.

In [ ]:
def search(query, k=5, filenames=None):
    """Top-k video windows for `query` across the registered videos, or across `filenames` only.

    `query` is a sentence, or a vector you already have (section 4.10). Each hit keeps its file, its VIOS
    sensor id and its window: as indexed (2025-01-01) and on the video's own VIOS timeline.
    An unregistered file name raises KeyError.
    """
    selected = list(sources) if filenames is None else list(filenames)
    if not selected:
        return []
    by_id = {sources[name]["sensor_id"]: name for name in selected}
    vector = query
    if isinstance(query, str):                                    # a sentence: encode it first, as in 4.3
        vector = api("POST", "/rtvi-embed/v1/generate_text_embeddings", timeout=60, json={
            "text_input": [query], "model": "cosmos-embed1-448p-anomaly-detection"})["data"][0]["embeddings"]
    these_videos = {"terms": {"sensor.id.keyword": list(by_id)}}
    data = api("POST", "/es/" + INDEX + "/_search", timeout=60, json={
        "size": k * 4,                                            # extra neighbors, so duplicates can go
        "_source": ["sensor.id", "timestamp", "end"],
        "query": {"bool": {"filter": [these_videos], "must": [{"nested": {
            "path": "llm.visionEmbeddings", "score_mode": "max",
            "query": {"knn": {"field": "llm.visionEmbeddings.vector", "query_vector": vector, "k": k * 4,
                              "filter": these_videos, "num_candidates": max(8 * k, 40)}}}}]}},
    })

    def when(iso):
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))

    def iso(moment):
        return moment.isoformat(timespec="milliseconds").replace("+00:00", "Z")

    anchor, seen, out = when(EMBEDDING_ANCHOR), set(), []
    for hit in data["hits"]["hits"]:
        src = hit["_source"]
        key = (src["sensor"]["id"], src["timestamp"], src["end"])  # video + window, never time alone
        if key in seen:
            continue
        seen.add(key)
        name = by_id[src["sensor"]["id"]]
        start, end = when(src["timestamp"]) - anchor, when(src["end"]) - anchor   # offsets into the video
        video_start = when(sources[name]["creation_time"])                         # where VIOS starts this video
        out.append({"filename": name, "sensor_id": src["sensor"]["id"], "score": round(hit["_score"], 4),
                    "embedding_start": src["timestamp"], "embedding_end": src["end"],
                    "start": iso(video_start + start), "end": iso(video_start + end),
                    "start_seconds": start.total_seconds(), "end_seconds": end.total_seconds()})
    return out[:k]


def show_results(hits):
    """One line per hit: rank, score, file, and the window in seconds from the start of the video."""
    if not hits:
        print("    (no hits)")
    for rank, hit in enumerate(hits, 1):
        print(f'[{rank}] {hit["score"]:.4f}  {hit["filename"]:18} '
              f'{hit["start_seconds"]:6.1f} to {hit["end_seconds"]:6.1f} s')


QUERY = "two vehicles collide at an intersection"
hits = search(QUERY)
show_results(hits)

## 4.5 Compare with an unrelated query

**Predict before you run.** The next cell searches the same four videos for "a polar bear in a snowstorm". Write
down your answers first. Does the search return nothing, or the closest chunks it has? How far below the collision
scores will these scores be?

In [ ]:
CONTROL = "a polar bear in a snowstorm"
control_hits = search(CONTROL)
show_results(control_hits)

labtools.plot_scores({
    QUERY: [hit["score"] for hit in hits],
    CONTROL: [hit["score"] for hit in control_hits],
})

**Check your prediction.** The search still returns five windows. A nearest-neighbor search always ranks the
closest vectors it has, even when nothing matches. The chart shows both score curves. In the embedding space, the
two queries look like this:

<img src="../images/course/embedding-knn.gif" width="960" alt="Cosmos-Embed1 places the query 'two vehicles collide at an intersection' and the control 'a polar bear in a snowstorm' among the 5-second chunks of the four videos, and each gets its five nearest chunks. A chart of similarity by rank shows the query's curve well above the control's, and the caption reads 'Nearest neighbors always exist: the gap between the curves is the signal.'">

The scores in the figure are examples. Read your own from the chart that the previous cell drew.

Scores are similarities, not probabilities. The dashed line at 0.5 only helps you read the chart. It is not a
threshold. Scores change with the videos, the wording and the model. If you pick a threshold here, measure it again
before you use it on other footage or another model. The gap between the two curves is the useful signal, and only
the video confirms a hit.

A query can also match how a scene looks without matching the action you asked for. So the next step is to watch
the window.

## 4.6 Watch the window

Each hit carries its window twice:

- `embedding_start` and `embedding_end` are times in the embedding index, where every video starts at 2025-01-01.
- `start` and `end` are the same moments on the video's VIOS timeline, which starts at 2025-01-02.

VIOS cuts clips only from its own timeline, so `search` converts each time for you: it measures the offset from
2025-01-01, then adds that offset to the video's VIOS start time. Use `start` and `end` whenever you ask VIOS for a
clip.

<img src="../images/course/two-anchors.gif" width="960" alt="Two time axes from 0 to 180 s: the embedding index, where every file sits at 2025-01-01, and the VIOS timeline of rain_eval.mp4, which starts at 2025-01-02. A hit at 2025-01-01T00:01:55 sent to VIOS as it is returns an empty MP4 with no error, while the same 115 s offset added to the file's creation_time gives 2025-01-02T00:01:55 and the clip, and the caption reads 'Convert the offset, never the date.'">

With the window on the VIOS timeline, `course_setup.clip_bytes` cuts exactly those seconds out of VIOS. The next cell
plays the top hit of the collision query.

In [ ]:
def clip_for(hit):
    """The hit's window as MP4 bytes, cut from VIOS on the video's own timeline."""
    return course_setup.clip_bytes(hit["sensor_id"], hit["start"], hit["end"])


if not hits:
    raise RuntimeError("the collision query returned no hits: check the embedding counts in 4.2")
show_results(hits[:1])
labtools.show_video(clip_for(hits[0]))

Does the window show a collision, two vehicles coming close, or ordinary traffic? Describe what you see before you
decide what it means.

If the action starts just before the window or ends just after it, run the next cell. It cuts the same window
again with up to 3 more seconds on each side, and prints the new window.

In [ ]:
WIDEN_S = 3                                                        # seconds to add on each side of the window
top_hit = hits[0]
length_s = sources[top_hit["filename"]]["duration_seconds"]
before = min(WIDEN_S, top_hit["start_seconds"])                    # never before the start of the video
after = max(0.0, min(WIDEN_S, length_s - top_hit["end_seconds"]))  # never past its end
print(f'{top_hit["filename"]}: {top_hit["start_seconds"] - before:.1f} to {top_hit["end_seconds"] + after:.1f} s')
labtools.show_video(course_setup.clip_bytes(top_hit["sensor_id"],
                                            course_setup.iso_offset(top_hit["start"], -before),
                                            course_setup.iso_offset(top_hit["end"], after)))

## 4.7 Write queries like captions

Cosmos-Embed1 learned from pairs of video and caption. A caption names what is in the video in a few words, such as
"vehicle collision" or "a car stopped in the intersection", and it never asks for anything. A query works best in
the same form. People often type a request for a chatbot instead, such as "Show me the clips where two cars crash
into each other". Words like "show me" and "clips" name nothing in the video, yet they still move the vector.

**Your turn:** write two queries the way a caption would describe the moment: one for a collision and one for a
stalled vehicle. Use a few words that name what you would see, and no request. Replace the two `FIXME`s, then run
the check cell. It searches the four videos with your two queries and with the chatbot-style request, prints the top
3 windows of each, and plays the top window of each of your queries. Click the `...` under the cell for the
solution.

**Predict before you run.** Will your collision query and the request return the same windows?

In [ ]:
# TODO
REQUEST = "Show me the clips where two cars crash into each other."   # a request written for a chatbot (given)
COLLISION = FIXME     # a collision, as a short caption: a few words, no request
STALLED = FIXME       # a stalled vehicle, as a short caption

In [ ]:
# SOLUTION
REQUEST = "Show me the clips where two cars crash into each other."   # a request written for a chatbot (given)
COLLISION = "vehicle collision"                  # what a caption of the moment would say
STALLED = "a stalled car in the intersection"    # a stopped vehicle, named the same way

In [ ]:
# Check: two short queries in caption form; then compare them with the request and watch their top windows.
import re

REQUEST_WORDS = {"show", "find", "give", "please", "clip", "clips", "where", "want", "me"}
for name, query in (("COLLISION", COLLISION), ("STALLED", STALLED)):
    if not isinstance(query, str) or not query.strip():
        raise ValueError(f"{name} must be a string of a few words, such as \"a white car\"")
    words = re.findall(r"[a-z']+", query.lower())
    asks = sorted(set(words) & REQUEST_WORDS)
    if len(words) > 8 or "?" in query or asks:
        raise ValueError(f"{name} = {query!r} reads like a request ({', '.join(asks) or 'too long'}). Describe the "
                         "moment in a few words, the way a caption would, and run your # TODO cell again.")

for label, query in (("request", REQUEST), ("your collision query", COLLISION), ("your stalled query", STALLED)):
    print(f"\n{label}: {query!r}")
    show_results(search(query, k=3))

for query in (COLLISION, STALLED):
    best = search(query, k=1)
    if not best:
        raise RuntimeError("no windows returned: check the embedding counts in 4.2")
    print(f"\ntop window for {query!r}: {best[0]['filename']} {best[0]['start_seconds']:.1f} to "
          f"{best[0]['end_seconds']:.1f} s")
    labtools.show_video(clip_for(best[0]), width=480)
print("\nPASS: two queries in caption form. Watch both windows before you decide which wording works.")

Compare the three lists. A request and a caption-style query often return different windows, and the request's
scores mean less: part of its vector stands for words that describe no video. Then watch your two windows. Did the
collision query find a collision, and the stalled query a stopped car?

To narrow a search, add what a caption would add, such as "at night" or "in the rain", not an instruction. Change
one word at a time, and watch a window before you decide whether the change helped.

## 4.8 Measure the search against the labels

Watching a window tells you about one hit. To compare two queries, count. Each search video comes with labels: the
start and end of every collision and every stalled vehicle in it, in seconds from the start of the video. They are
in `/dli/task/data/ground_truth/search_videos.json`. The next cell scores three collision queries in two ways:

- **Top-5 precision:** how many of the 5 best windows overlap a labelled collision.
- **Collisions found:** how many of the 8 labelled collisions in the four videos at least one of the 10 best windows
  overlaps.

**Predict before you run.** Which query will score best: the request, the sentence from section 4.4, or your
collision query?

In [ ]:
# Score three collision queries against the labels: top-5 precision, and the labelled collisions the top 10 reach.
LABELS = TASK / "data/ground_truth/search_videos.json"
if not LABELS.is_file():
    raise FileNotFoundError(f"{LABELS} is missing: the course data did not load. Call the instructor.")
labels = {video["file"]: video["events"] for video in json.loads(LABELS.read_text())}
collisions = [(name, event) for name, events in labels.items() if name in sources
              for event in events if event["type"] == "collision"]


def overlaps(hit, event):
    """True when the hit's window and the labelled event share any time."""
    return hit["start_seconds"] < event["end_s"] and hit["end_seconds"] > event["start_s"]


def on_a_label(hit):
    return any(name == hit["filename"] and overlaps(hit, event) for name, event in collisions)


scored = {}
print(f"{'query':58} top-5 precision   collisions found")
for query in (REQUEST, QUERY, COLLISION):
    top10 = search(query, k=10)
    scored[query] = top10
    precision = sum(on_a_label(hit) for hit in top10[:5])
    found = sum(any(h["filename"] == name and overlaps(h, event) for h in top10) for name, event in collisions)
    print(f"{query[:56]:58} {precision}/5             {found} of {len(collisions)}")

In [ ]:
# The top-5 windows of your collision query that overlap no labelled collision. Watch the first one.
misses = [hit for hit in scored[COLLISION][:5] if not on_a_label(hit)]
print(f"top-5 windows for {COLLISION!r} that overlap no labelled collision: {len(misses)}")
show_results(misses)
if misses:
    labtools.show_video(clip_for(misses[0]), width=480)

**Check your prediction.** Compare the columns, not single scores. Then watch the window above, if there is one. A
window that matches no label is a false alarm, or a collision the labels missed, and only the footage tells you
which. People make labels, and an evaluation is only as good as its labels. Before you trust a number, watch a
sample of the windows it counted as wrong.

This is how you choose between two wordings, two models or two settings: the same labels, the same counts, and one
change at a time.

## 4.9 Search one file at a time

With one ranked list across four videos, one video can take all five places, and you learn nothing about the other
three. The `filenames` argument of `search` fixes that. For each name you pass, `search` looks up the
video's VIOS sensor id. Those ids become the `sensor.id.keyword` filter inside
the `knn` query, so the search ranks only those videos' windows.

**Your turn:** Complete `search_each_file(phrase, k)`. It runs `search` once per video, restricted to that one video,
and returns `{filename: hits}`. Replace the two `FIXME`s. Click the `...` under the cell for the solution.

**Predict before you run.** Which video's top hit will score highest for the collision query? Write it down, then
run the check cell. It prints `PASS` and one list per video, then plays the highest-scoring hit.

In [ ]:
# TODO
def search_each_file(phrase, k=3):
    """One ranked list per video: {filename: hits}, each list restricted to that one video."""
    results = {}
    for name in FIXME:                                        # every video in the collection, by file name
        results[name] = search(phrase, k=k, filenames=FIXME)  # keep this search inside the one video
    return results

In [ ]:
# SOLUTION
def search_each_file(phrase, k=3):
    """One ranked list per video: {filename: hits}, each list restricted to that one video."""
    results = {}
    for name in sources:                                        # the four videos from 4.1
        results[name] = search(phrase, k=k, filenames=[name])   # a one-name list: this video only
    return results

In [ ]:
# Check: four lists, each at most k long, each holding only its own video's windows.
condition_results = search_each_file(QUERY)
if set(condition_results) != set(sources):
    raise AssertionError(f"expected one list per video {sorted(sources)}, got {sorted(condition_results)}")
for name, file_hits in condition_results.items():
    leaked = sorted({h["filename"] for h in file_hits} - {name})
    if leaked:
        raise AssertionError(f"{name}: the filter let windows from {leaked} through; pass filenames=[name]")
    if len(file_hits) > 3:
        raise AssertionError(f"{name}: {len(file_hits)} hits returned for k=3")
if not any(condition_results.values()):
    raise RuntimeError("no video returned a window: check the embedding counts in 4.2")
print("PASS: one list per video, each with only its own windows.")

for name, file_hits in condition_results.items():
    print("\n" + name)
    show_results(file_hits)

top = max((h for file_hits in condition_results.values() for h in file_hits[:1]), key=lambda h: h["score"])
print(f"\nhighest top hit: {top['filename']} at {top['start_seconds']:.1f} s (score {top['score']:.4f}). "
      "Was that your prediction?")
labtools.show_video(course_setup.clip_bytes(top["sensor_id"], top["start"], top["end"]), width=480)

Watch the clip above before you decide anything. The four videos are not aligned frame by frame. A similar
background or the same offset in two videos does not mean the same event. Each video's top hit is a candidate to
watch, not a result.

## 4.10 Optional: search with a clip instead of a sentence

Cosmos-Embed1 embeds video as well as text, so a short clip can be the query: "find more moments like this one".
RT-Embed has no search route of its own. You make the query vector with RT-Embed, then run the same kNN search in
Elasticsearch.

The next cell cuts seconds 2 to 7 of `traffic_short`, the clip you added to VIOS in notebook 2. The two cars in it
collide at about 3.5 to 4 seconds. The cell then makes one vector for the whole clip with two calls to RT-Embed:

1. `POST /rtvi-embed/v1/files` uploads the clip. RT-Embed returns a new file id.
2. `POST /rtvi-embed/v1/generate_video_embeddings` with that id and `chunk_duration` set to 0 returns one embedding
   for the whole clip, instead of one per 5 seconds.

RT-Embed publishes every embedding it makes to Kafka, so this query vector also reaches Elasticsearch, the same way
the collection's vectors did. The index name carries the date of the `creation_time` you send with the RT-Embed
upload. The cell uses 2026-11-01, so the vector lands in `mdx-embed-filtered-2026-11-01` and stays out of the
2025-01-01 index you search. Then the cell deletes the uploaded file. Look for one chunk with 768 values.

In [ ]:
SHORT_START = "2025-01-01T00:00:00.000Z"   # where notebook 2 started traffic_short's VIOS timeline
QUERY_DATE = "2026-11-01T00:00:00.000Z"    # a date of its own, away from the 2025-01-01 index you search

short = course_setup.find_sensor("traffic_short")
if short is None:
    raise RuntimeError("VIOS has no traffic_short source. Add it in notebook 2 first.")
example = course_setup.clip_bytes(short["sensorId"], course_setup.iso_offset(SHORT_START, 2),
                                  course_setup.iso_offset(SHORT_START, 7))
labtools.show_video(example, width=480)

file_id = api("POST", "/rtvi-embed/v1/files", timeout=120,
              files={"file": ("query_clip.mp4", example, "video/mp4")},
              data={"purpose": "vision", "media_type": "video", "creation_time": QUERY_DATE})["id"]
try:
    result = api("POST", "/rtvi-embed/v1/generate_video_embeddings", timeout=300, json={
        "id": file_id, "model": "cosmos-embed1-448p-anomaly-detection",
        "chunk_duration": 0,                  # 0: no chunking, one embedding for the whole clip
    })
finally:
    api("DELETE", "/rtvi-embed/v1/files/" + file_id, timeout=30)   # the file goes; its vector stays indexed

chunks = result["chunk_responses"]
clip_vector = chunks[0]["embeddings"]
print("RT-Embed file id:", file_id)
print("chunks:", len(chunks), "   values in the vector:", len(clip_vector))

Now run the same search as in 4.4, with the clip's vector in place of a sentence. The next cell searches the four
videos and plays the top hit.

In [ ]:
clip_hits = search(clip_vector)       # the kNN search from 4.4, with a video vector instead of a text vector
show_results(clip_hits)
if not clip_hits:
    raise RuntimeError("the clip query returned no windows: check the embedding counts in 4.2")
labtools.show_video(clip_for(clip_hits[0]), width=480)

Compare these windows with the hits of the collision sentence in 4.4. Do the same moments come back? Does the top
window you just watched show a collision?

An image can be a query too. Upload it with `media_type` set to `image`, and RT-Embed turns it into a still clip
before it embeds it. The search works the same way.

## Check your understanding

1. The polar-bear query returned five windows. Is there a polar bear in the footage?
2. A hit's `embedding_start` is `2025-01-01T00:01:55Z`. You ask VIOS for a clip at that time and get an empty body.
   What went wrong?
3. Why does `search` remove duplicates by (sensor id, start, end) and not by (start, end)?
4. The `sensor.id.keyword` filter appears twice in the request: in `bool.filter` and inside the `knn` query.
   Which one keeps a one-video search from coming back short, and why?
5. A colleague searches for "Please find me any clips where a car breaks down". What would you change, and why?

<details>
<summary>Answers</summary>

1. No. A nearest-neighbor search always returns the `k` closest vectors, whether or not the scene exists. The gap
   between the control scores and the real query's scores is the signal, and only watching the clip confirms a hit.
2. The time is still on the index's date. Uploaded videos are indexed at 2025-01-01, but the search videos' VIOS
   timelines start at 2025-01-02. Take the offset from 2025-01-01 (115 s here) and add it to the video's
   `creation_time`, as `search` does for `start` and `end`.
3. All four videos start at offset 0 and run about 180 s, so two videos can match at the same offset. Removing
   duplicates by time alone would drop a real hit from another video.
4. The one inside `knn`. It limits the neighbors during the vector search, so all `k` come from the selected
   videos. A `bool.filter` is applied only after the neighbors are chosen. On its own it could leave a one-video
   search with fewer than `k` windows, or none, because the nearest vectors belonged to other videos.
5. Write it the way a caption would describe the moment, such as "a stalled car". Cosmos-Embed1 learned from pairs of
   video and caption, and words like "please find me" and "clips" name nothing in the video. Then measure both
   wordings against labels, as in section 4.8.

</details>

## Next

Open [Part 1 Assessment](assessment.ipynb). You get a new 30-second clip and use the building blocks from notebooks
1 to 4 on your own. One of the tasks is a text search of these four videos.

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **You are done with this step when:** the check cell of the exercise in 4.9 prints `PASS`.
>
> **Next: [Part 1 Assessment](assessment.ipynb)** &middot; Part 1 &middot; about 25 min  
> On your own: use the building blocks on a new 30-second clip, answer eight questions, and submit.
>
> [Back to Home](../00_Home.ipynb)


## Instructor only: prepare the collection by hand

Learners skip this section. At launch, the `course-prep` service uploads the four search videos with
`course_setup.ingest_file` and writes the registry and the status file. On a devbox without that service:

1. Run the next cell. It imports `prepare_search_collection()` from `/dli/prep/prepare_course.py`, the same code
   course-prep runs, and uploads nothing.
2. Call `prepare_search_collection(task_dir='/dli/task')` from a new cell or a terminal. It uploads the four videos
   one after another and waits until RT-Embed has indexed all of them.
3. Run sections 4.1 and 4.2 again.

It writes `data/search_sources.json` and `data/course_prep_status.json` in course-prep's format. The state becomes
`OK` only when every video is searchable.

In [ ]:
# INSTRUCTOR ONLY. Imports prepare_search_collection() from the course-prep script and calls nothing.
import sys

sys.path.insert(0, "/dli/prep")   # in the lab image (lab/Dockerfile: COPY lab/prep /dli/prep), also on its PYTHONPATH
import prepare_course             # ModuleNotFoundError here means a lab image older than the /dli/prep layout: rebuild it

# The same function course-prep runs at launch: it uploads every search video that is not searchable yet (not in
# VIOS, or its search records are gone), waits until Cosmos-Embed1 covers each one, and writes
# data/search_sources.json and data/course_prep_status.json (state OK only when every video is searchable; FAILED
# and an error otherwise).
prepare_search_collection = prepare_course.prepare_search_collection

print(f"imported prepare_search_collection() from {prepare_course.__file__}; nothing was uploaded.")
print("Call it from a new cell or a terminal: prepare_search_collection(task_dir='/dli/task')")

<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>